In [1]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree

#### Constants

In [2]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')

str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

str_dirname_output = './output'

flt_factor_24_to_72 = 2.36

# id cols
list_cols_id = [
    'request_datetime',
    'accountid',
    'bitdebtor',
]

# ivs
list_cols_structure = [
    'ENG-wtd_avg',
    's071b__tu', # Number of non-medical third party collections opened in past 36 months
    'g215b__tu', # Number of non-medical third party collections with balance > $0
    's068b__tu', # Number of non-medical third party collections
    'amtfinanced__app',
]

# col w institutions
list_cols_inst = [
    'str_institution__tu_pmthx',
]

# targets
list_str_target = [
    'Early_Pay_Delinquency_15_60_Flag',
    'Early_Pay_Delinquency_30_90_Flag',
    'Early_Pay_Delinquency_30_180_Flag',
    'Early_Pay_Delinquency_30_360_Flag',
    'Early_Pay_Delinquency_60_720_Flag',
#     'loss_at_60',
#     'loss_at_180',
#     'loss_at_360',
    'loss_at_720',
]

# list cols
list_cols = list_cols_id + list_cols_structure + list_cols_inst + list_str_target
int_len = len(list_cols)
print(f'Importing {int_len} columns')

list_str_inst = [
    # from ben: 2025-02-21
    'CURRENT',
    'SELF',
    # key words
    'CHIME-STRIDE',
    'CHIMEFINAL',
    # from dustin: 2025-02-24
    'SELF FIN',
    'SELF/LEAD',
    'SELFINC/LEAD',
    'SBNASELFLNDR',
    'SBNA SELF',
    'CHIME',
    'CLEO',
    'CLEO AI',
    'VARO',
    'ATLAS',
    'ATLCAPBKSELF',
    'POSSIBLE',
    'POSSIBLE FIN',
    'KIKOFF',
    'SUPER.COM',
    'STEP',
    'STEP MOBILE',
    'BRIGHT',
    'BRIGHT BLDR',
    'FIG TECH INC',
    'SELF/RENT',
    'SELFBILLSE',
    'PROGRESSRES',
    'FLEX',
    'FLEXFINANCE',
]

Project: 20250221-credit-builder-analysis
Task: 06_collections
Importing 15 columns


#### Output dir

In [3]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

#### Import data

In [4]:
str_filename = 'df.gzip'
str_uri = f's3://20241112-simple-model-test/08_prep_data/{str_filename}'
df = pd.read_parquet(
    str_uri,
    columns=list_cols,
)
# sort
df.sort_values(by='request_datetime', ascending=True, inplace=True)
df

,request_datetime,accountid,bitdebtor,ENG-wtd_avg,s071b__tu,g215b__tu,s068b__tu,amtfinanced__app,str_institution__tu_pmthx,Early_Pay_Delinquency_15_60_Flag,Early_Pay_Delinquency_30_90_Flag,Early_Pay_Delinquency_30_180_Flag,Early_Pay_Delinquency_30_360_Flag,Early_Pay_Delinquency_60_720_Flag,loss_at_720
0,2021-07-26 16:29:29.3903686,5702434,1,0.616667,NaN,NaN,NaN,14292.30,"['CAPITAL ONE', 'FST PREMIER', 'VERIDIAN CU', ...",0,0,0,0,0,0.00
1,2021-07-26 16:39:34.1121025,5714239,1,NaN,1.0,2.0,2.0,24771.37,"['MTN AMER CU', 'MTN AMER CU', 'MID CITY FIN']",1,0,1,1,1,7827.16
2,2021-07-26 16:48:39.3211104,5713063,1,NaN,NaN,NaN,NaN,17188.00,"['NAVIENT', 'NLS', 'FST PREMIER', 'SHELLPOINT'...",0,0,0,0,0,0.00
3,2021-07-27 09:02:35.3300974,5713732,1,1.000000,NaN,NaN,0.0,26127.42,"['CAPITAL ONE', 'CB INDIGO', 'CHIME-STRIDE', '...",0,0,1,1,1,0.00
4,2021-07-27 09:18:12.2190097,5715634,1,1.000000,NaN,NaN,NaN,18554.36,"['ARIZ FED CU', 'ARIZ FED CU']",0,0,0,0,0,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94472,2024-11-26 06:16:16+00:00,8420588,1,0.030636,4.0,4.0,4.0,30550.82,"['FST PREMIER', 'CONNS', 'STATECU', 'OPENSKY C...",1,0,0,0,0,0.00
94473,2024-11-26 06:21:44+00:00,8401043,1,0.367073,NaN,NaN,NaN,27678.30,"['CAPITAL ONE', 'CAPITAL ONE', 'NAVY FCU', 'DO...",1,1,1,1,0,0.00
94474,2024-11-26 06:25:09+00:00,8414683,1,NaN,2.0,2.0,2.0,18742.43,"['LENDMARK', 'LENDMARK', 'CB INDIGO', 'LENDMAR...",1,0,0,0,0,0.00
94475,2024-11-26 06:32:28+00:00,8359085,1,0.006585,1.0,0.0,1.0,23587.85,"['MAF AUTO', 'EDFINANCIAL', 'EDFINANCIAL', 'ED...",0,0,0,0,0,0.00


#### Fillna

In [5]:
list_cols = [
    's071b__tu',
    'g215b__tu',
    's068b__tu',
]
for col in tqdm(list_cols):
    df[col] = df[col].fillna(999)

100%|██████████| 3/3 [00:00<00:00, 866.59it/s]


#### Get Gen 12 predictions

In [6]:
list_cols = [
    'accountid',
    'bitdebtor',
    'request_datetime',
    'ad',
    'pd',
    'lgd',
]

str_filename = 'df_clean_w_pred.gzip'
str_uri = f's3://{str_project}/01_gen12_predictions/{str_filename}'
df_tmp = pd.read_parquet(
    str_uri,
    columns=list_cols,
)
df_tmp['ecnl'] = df_tmp['pd'] * df_tmp['lgd'] * flt_factor_24_to_72
# rename
dict_rename =  {
    'ad': 'gen12_ad',
    'pd': 'gen12_pd',
    'lgd': 'gen12_lgd',
    'ecnl': 'gen12_ecnl',
}
df_tmp.rename(columns=dict_rename, inplace=True)
# join
df = pd.merge(
    left=df,
    right=df_tmp,
    on=['accountid','bitdebtor','request_datetime'],
    how='left',
)
# show
#df

#### Get Gen 13 predictions

In [7]:
list_cols = [
    'accountid',
    'bitdebtor',
    'request_datetime',
    'pd',
    'lgd',
]

str_filename = 'df_clean_w_pred.gzip'
str_uri = f's3://{str_project}/02_gen13_predictions/{str_filename}'
df_tmp = pd.read_parquet(
    str_uri,
    columns=list_cols,
)
df_tmp['ecnl'] = df_tmp['pd'] * df_tmp['lgd'] * flt_factor_24_to_72
# rename
dict_rename =  {
    'pd': 'gen13_pd',
    'lgd': 'gen13_lgd',
    'ecnl': 'gen13_ecnl',
}
df_tmp.rename(columns=dict_rename, inplace=True)
# join
df = pd.merge(
    left=df,
    right=df_tmp,
    on=['accountid','bitdebtor','request_datetime'],
    how='left',
)
# show
#df

#### Make a tag at the account level if pmt hx is bad

In [8]:
df_tmp = df.groupby('accountid', as_index=False).agg({
    'ENG-wtd_avg': 'mean',
})
df_tmp['tag'] = df_tmp['ENG-wtd_avg'].apply(
    lambda x: 1 if x < 0.5 else 0,
)
flt_mn = df_tmp['tag'].mean()
print(f'Proportion of accounts with bad pmt hx: {flt_mn:0.4f}')
df_tmp = df_tmp[df_tmp['tag'] == 1].copy()
df_tmp['accountid'] = df_tmp['accountid'].astype(int)
list_accountid = list(df_tmp['accountid'])
# creat tag
df['bad_pmt_hx'] = df['accountid'].apply(
    lambda x: 1 if x in list_accountid else 0,
)
# show
#df

Proportion of accounts with bad pmt hx: 0.2365


#### Tag for s071b__tu

In [9]:
str_col = 's071b__tu'
df_tmp = df.groupby('accountid', as_index=False).agg({
    str_col: 'max',
})
df_tmp['tag'] = df_tmp[str_col].apply(
    lambda x: 1 if x > 1 else 0,
)
flt_mn = df_tmp['tag'].mean()
print(f'Proportion of accounts with > 1 {str_col}: {flt_mn:0.4f}')
df_tmp = df_tmp[df_tmp['tag'] == 1].copy()
df_tmp['accountid'] = df_tmp['accountid'].astype(int)
list_accountid = list(df_tmp['accountid'])
# creat tag
df[f'bad_{str_col}'] = df['accountid'].apply(
    lambda x: 1 if x in list_accountid else 0,
)
# show
#df

Proportion of accounts with > 1 s071b__tu: 0.7609


#### Tag for g215b__tu

In [10]:
str_col = 'g215b__tu'
df_tmp = df.groupby('accountid', as_index=False).agg({
    str_col: 'max',
})
df_tmp['tag'] = df_tmp[str_col].apply(
    lambda x: 1 if x > 1 else 0,
)
flt_mn = df_tmp['tag'].mean()
print(f'Proportion of accounts with > 1 {str_col}: {flt_mn:0.4f}')
df_tmp = df_tmp[df_tmp['tag'] == 1].copy()
df_tmp['accountid'] = df_tmp['accountid'].astype(int)
list_accountid = list(df_tmp['accountid'])
# creat tag
df[f'bad_{str_col}'] = df['accountid'].apply(
    lambda x: 1 if x in list_accountid else 0,
)
# show
#df

Proportion of accounts with > 1 g215b__tu: 0.8236


#### Tag for s068b__tu

In [11]:
str_col = 's068b__tu'
df_tmp = df.groupby('accountid', as_index=False).agg({
    str_col: 'max',
})
df_tmp['tag'] = df_tmp[str_col].apply(
    lambda x: 1 if x > 1 else 0,
)
flt_mn = df_tmp['tag'].mean()
print(f'Proportion of accounts with > 1 {str_col}: {flt_mn:0.4f}')
df_tmp = df_tmp[df_tmp['tag'] == 1].copy()
df_tmp['accountid'] = df_tmp['accountid'].astype(int)
list_accountid = list(df_tmp['accountid'])
# creat tag
df[f'bad_{str_col}'] = df['accountid'].apply(
    lambda x: 1 if x in list_accountid else 0,
)
# show
#df

Proportion of accounts with > 1 s068b__tu: 0.7490


#### Get chargeoff severity

In [12]:
df['co_at_720'] = df['loss_at_720'] / df['amtfinanced__app']
# make min 0
df['co_at_720'] = df['co_at_720'].clip(lower=0)
# show
#df

#### Make 60+ in 720 and positive net charge off

In [13]:
df['60_plus_pos_co_at_720'] = df.apply(
    lambda x: 1 if (x['Early_Pay_Delinquency_60_720_Flag'] == 1) and (x['co_at_720'] > 0) else 0,
    axis=1,
)
# show
#df

#### Create a new column that is a list

In [14]:
df['list_institutions'] = df['str_institution__tu_pmthx'].apply(
    lambda x: eval(x.replace('nan','None')),
)
df['list_institutions'] = df['list_institutions'].apply(
    lambda x: [] if x is None else x,
)
# show
#df

#### Get the unique names for institutions

In [15]:
list_str_inst_flat = list(df['list_institutions'].explode().dropna())
list_str_inst_flat = list(dict.fromkeys(list_str_inst_flat))
int_len = len(list_str_inst_flat)
print(f'There are {int_len} unique institutions')

There are 9533 unique institutions


#### Find any with keywords

In [16]:
list_cols = [col for col in list_str_inst_flat if 'chime' in col.lower()]
list_cols

['CHIME-STRIDE', 'CHIMEFINAL']

#### Create tag for institutions

In [17]:
list_str_col_new = []
for str_inst in tqdm(list_str_inst):
    str_col_new = f'{str_inst}_tag'
    df[str_col_new] = df['list_institutions'].apply(
        lambda x: 1 if str_inst in x else 0,
    )
    list_str_col_new.append(str_col_new)

# show
df

100%|██████████| 29/29 [00:01<00:00, 20.31it/s]


,request_datetime,accountid,bitdebtor,ENG-wtd_avg,s071b__tu,g215b__tu,s068b__tu,amtfinanced__app,str_institution__tu_pmthx,Early_Pay_Delinquency_15_60_Flag,...,STEP_tag,STEP MOBILE_tag,BRIGHT_tag,BRIGHT BLDR_tag,FIG TECH INC_tag,SELF/RENT_tag,SELFBILLSE_tag,PROGRESSRES_tag,FLEX_tag,FLEXFINANCE_tag
0,2021-07-26 16:29:29.3903686,5702434,1,0.616667,999.0,999.0,999.0,14292.30,"['CAPITAL ONE', 'FST PREMIER', 'VERIDIAN CU', ...",0,...,0,0,0,0,0,0,0,0,0,0
1,2021-07-26 16:39:34.1121025,5714239,1,NaN,1.0,2.0,2.0,24771.37,"['MTN AMER CU', 'MTN AMER CU', 'MID CITY FIN']",1,...,0,0,0,0,0,0,0,0,0,0
2,2021-07-26 16:48:39.3211104,5713063,1,NaN,999.0,999.0,999.0,17188.00,"['NAVIENT', 'NLS', 'FST PREMIER', 'SHELLPOINT'...",0,...,0,0,0,0,0,0,0,0,0,0
3,2021-07-27 09:02:35.3300974,5713732,1,1.000000,999.0,999.0,0.0,26127.42,"['CAPITAL ONE', 'CB INDIGO', 'CHIME-STRIDE', '...",0,...,0,0,0,0,0,0,0,0,0,0
4,2021-07-27 09:18:12.2190097,5715634,1,1.000000,999.0,999.0,999.0,18554.36,"['ARIZ FED CU', 'ARIZ FED CU']",0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94472,2024-11-26 06:16:16+00:00,8420588,1,0.030636,4.0,4.0,4.0,30550.82,"['FST PREMIER', 'CONNS', 'STATECU', 'OPENSKY C...",1,...,0,0,0,0,0,0,0,0,0,0
94473,2024-11-26 06:21:44+00:00,8401043,1,0.367073,999.0,999.0,999.0,27678.30,"['CAPITAL ONE', 'CAPITAL ONE', 'NAVY FCU', 'DO...",1,...,0,0,0,0,0,0,0,0,0,0
94474,2024-11-26 06:25:09+00:00,8414683,1,NaN,2.0,2.0,2.0,18742.43,"['LENDMARK', 'LENDMARK', 'CB INDIGO', 'LENDMAR...",1,...,0,0,0,0,0,0,0,0,0,0
94475,2024-11-26 06:32:28+00:00,8359085,1,0.006585,1.0,0.0,1.0,23587.85,"['MAF AUTO', 'EDFINANCIAL', 'EDFINANCIAL', 'ED...",0,...,0,0,0,0,0,0,0,0,0,0


#### Tag if there was an institution of interest

In [18]:
df['sum'] = df[list_str_col_new].sum(axis=1)
df['has_inst_tag'] = df['sum'].apply(
    lambda x: 1 if x > 0 else 0,
)
flt_mn = df['has_inst_tag'].mean()
print(f'Proportion has tag: {flt_mn:0.4f}')
# show
#df

Proportion has tag: 0.2053


#### Save to s3

In [19]:
%%time

str_filename = 'df_institutions.gzip'
str_uri = f's3://{str_project}/{str_task}/{str_filename}'
df.to_parquet(
    str_uri,
    compression='gzip',
)

CPU times: user 1.74 s, sys: 72.1 ms, total: 1.82 s
Wall time: 2.03 s


#### ECNL by chime (no targets yet because we are using newer data)

In [20]:
df_tmp = df.groupby(by='accountid', as_index=False).agg({
    'has_inst_tag': 'max',
    'bad_s071b__tu': 'max',
    'bad_g215b__tu': 'max',
    'bad_s068b__tu': 'max',
    'gen12_ad': 'mean',
    'gen12_pd': 'mean',
    'gen12_lgd': 'mean',
    'gen13_pd': 'mean',
    'gen13_lgd': 'mean',
})
# gen ecnls
df_tmp['gen12_ecnl'] = df_tmp['gen12_pd'] * df_tmp['gen12_lgd'] * flt_factor_24_to_72
df_tmp['gen13_ecnl'] = df_tmp['gen13_pd'] * df_tmp['gen13_lgd'] * flt_factor_24_to_72
# group by has_inst_tag
df_tmp = df_tmp.groupby(by='has_inst_tag', as_index=False).agg({
    'bad_s071b__tu': 'mean',
    'bad_g215b__tu': 'mean',
    'bad_s068b__tu': 'mean',
    'gen12_ad': 'mean',
    'gen12_pd': 'mean',
    'gen12_lgd': 'mean',
    'gen12_ecnl': 'mean',
    'gen13_pd': 'mean',
    'gen13_lgd': 'mean',
    'gen13_ecnl': 'mean',
})
# map
dict_map = {
    1: 'Yes',
    0: 'No',
}
df_tmp['has_inst_tag'] = df_tmp['has_inst_tag'].map(dict_map)

# save
str_filename = 'df_pivot_all_scores.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df_tmp.to_csv(str_local_path, index=False)
# show
df_tmp

,has_inst_tag,bad_s071b__tu,bad_g215b__tu,bad_s068b__tu,gen12_ad,gen12_pd,gen12_lgd,gen12_ecnl,gen13_pd,gen13_lgd,gen13_ecnl
0,No,0.754518,0.821154,0.737763,0.428422,0.243423,0.654586,0.378384,0.440916,0.352702,0.371948
1,Yes,0.781728,0.831550,0.786118,0.459883,0.259462,0.660201,0.406832,0.495680,0.353988,0.417104


#### Rm too new so we can look at targets

In [21]:
df = df[df['request_datetime'] <= '2022-11-26'].copy()
df

,request_datetime,accountid,bitdebtor,ENG-wtd_avg,s071b__tu,g215b__tu,s068b__tu,amtfinanced__app,str_institution__tu_pmthx,Early_Pay_Delinquency_15_60_Flag,...,BRIGHT_tag,BRIGHT BLDR_tag,FIG TECH INC_tag,SELF/RENT_tag,SELFBILLSE_tag,PROGRESSRES_tag,FLEX_tag,FLEXFINANCE_tag,sum,has_inst_tag
0,2021-07-26 16:29:29.3903686,5702434,1,0.616667,999.0,999.0,999.0,14292.30,"['CAPITAL ONE', 'FST PREMIER', 'VERIDIAN CU', ...",0,...,0,0,0,0,0,0,0,0,0,0
1,2021-07-26 16:39:34.1121025,5714239,1,NaN,1.0,2.0,2.0,24771.37,"['MTN AMER CU', 'MTN AMER CU', 'MID CITY FIN']",1,...,0,0,0,0,0,0,0,0,0,0
2,2021-07-26 16:48:39.3211104,5713063,1,NaN,999.0,999.0,999.0,17188.00,"['NAVIENT', 'NLS', 'FST PREMIER', 'SHELLPOINT'...",0,...,0,0,0,0,0,0,0,0,0,0
3,2021-07-27 09:02:35.3300974,5713732,1,1.000000,999.0,999.0,0.0,26127.42,"['CAPITAL ONE', 'CB INDIGO', 'CHIME-STRIDE', '...",0,...,0,0,0,0,0,0,0,0,1,1
4,2021-07-27 09:18:12.2190097,5715634,1,1.000000,999.0,999.0,999.0,18554.36,"['ARIZ FED CU', 'ARIZ FED CU']",0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41387,2022-11-25 23:17:46+00:00,6449613,1,0.464643,999.0,999.0,0.0,22164.00,"['CREDITONEBNK', 'CREDITONEBNK', 'CAPITAL ONE'...",1,...,0,0,0,0,0,0,0,0,0,0
41388,2022-11-25 23:21:35+00:00,6436672,1,0.771429,999.0,999.0,999.0,32481.00,"['U S AUTO CR', 'ASCENT AU FI']",0,...,0,0,0,0,0,0,0,0,0,0
41389,2022-11-25 23:26:06+00:00,6448141,1,0.083004,4.0,4.0,4.0,25278.37,"['LENDMARK', 'SANTANDER', 'NC FINANCIAL', 'EDF...",1,...,0,0,0,0,0,0,0,0,0,0
41390,2022-11-25 23:49:40+00:00,6439773,1,NaN,999.0,999.0,999.0,21160.14,"['TAB/SUNBIT', 'TBOM/FORTIVA', 'TBOM/CONTFIN',...",0,...,0,0,0,0,0,0,0,0,0,0


#### ECNLs, Delinquency, and CO severity

In [22]:
# ecnl at account level
df_tmp = df.groupby(by='accountid', as_index=False).agg({
    'has_inst_tag': 'max',
    'gen12_ad': 'mean',
    'gen12_pd': 'mean',
    'gen12_lgd': 'mean',
    'gen13_pd': 'mean',
    'gen13_lgd': 'mean',
    'co_at_720': 'first',
    '60_plus_pos_co_at_720': 'first',
    'Early_Pay_Delinquency_60_720_Flag': 'first',
    'bad_s071b__tu': 'max',
    'bad_g215b__tu': 'max',
    'bad_s068b__tu': 'max',
})
# get ecnl
df_tmp['gen12_ecnl_24'] = df_tmp['gen12_pd'] * df_tmp['gen12_lgd']
df_tmp['gen13_ecnl_24'] = df_tmp['gen13_pd'] * df_tmp['gen13_lgd']
# group
list_cols = [
    'has_inst_tag',
    'bad_s071b__tu',
    'bad_g215b__tu',
    'bad_s068b__tu',
]
df_tmp = df_tmp.groupby(by=list_cols, as_index=False).agg({
    'gen12_ad': 'mean',
    'gen12_pd': 'mean',
    'gen12_lgd': 'mean',
    'gen13_pd': 'mean',
    'gen13_lgd': 'mean',
    'gen12_ecnl_24': 'mean',
    'gen13_ecnl_24': 'mean',
    'co_at_720': 'mean',
    '60_plus_pos_co_at_720': 'mean',
    'Early_Pay_Delinquency_60_720_Flag': 'mean',
})

# convert back to Yes No
dict_map = {
    1: 'Yes',
    0: 'No',
}
list_cols = [
    'has_inst_tag',
    'bad_s071b__tu',
    'bad_g215b__tu',
    'bad_s068b__tu',
]
for col in tqdm(list_cols):
    df_tmp[col] = df_tmp[col].map(dict_map)

# save
str_filename = 'df_pivot_subset_targets_and_predictions.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df_tmp.to_csv(str_local_path, index=False)

# show
df_tmp

100%|██████████| 4/4 [00:00<00:00, 2626.36it/s]


,has_inst_tag,bad_s071b__tu,bad_g215b__tu,bad_s068b__tu,gen12_ad,gen12_pd,gen12_lgd,gen13_pd,gen13_lgd,gen12_ecnl_24,gen13_ecnl_24,co_at_720,60_plus_pos_co_at_720,Early_Pay_Delinquency_60_720_Flag
0,No,No,No,No,0.491046,0.277851,0.657011,0.481986,0.362613,0.183412,0.175879,0.177529,0.272747,0.475928
1,No,No,No,Yes,0.451003,0.244940,0.655189,0.454354,0.350916,0.161188,0.160835,0.127377,0.220238,0.413690
2,No,No,Yes,No,0.370445,0.216408,0.668966,0.420370,0.322712,0.146377,0.138007,0.131197,0.212717,0.426590
3,No,No,Yes,Yes,0.494595,0.263328,0.653121,0.468694,0.363818,0.172837,0.171394,0.171549,0.264481,0.450273
4,No,Yes,No,Yes,0.460400,0.247479,0.646264,0.476214,0.348801,0.160415,0.167525,0.130054,0.222222,0.458333
5,No,Yes,Yes,No,0.439592,0.263327,0.661060,0.435347,0.352178,0.174894,0.155307,0.141920,0.231216,0.407021
6,No,Yes,Yes,Yes,0.459828,0.266487,0.657228,0.466349,0.353072,0.176170,0.167100,0.152860,0.244514,0.440502
7,Yes,No,No,No,0.524888,0.306164,0.662959,0.546475,0.362195,0.204296,0.198991,0.245741,0.358115,0.588482
8,Yes,No,No,Yes,0.455395,0.257548,0.656482,0.482973,0.349891,0.169127,0.170902,0.214703,0.307692,0.538462
9,Yes,No,Yes,No,0.401103,0.249935,0.672516,0.481328,0.318899,0.169669,0.154561,0.211109,0.327103,0.588785
